# 导入工具和库

In [11]:
import sys
import os
import random

import numpy as np
import torch
import pandas as pd

from argparse import Namespace
from pathlib import Path


# 将项目的父目录添加到Python模块搜索路径中，以便导入项目中的自定义模块。
parent = Path(os.path.abspath("")).resolve().parents[0]
if parent not in sys.path:
    sys.path.insert(0, str(parent))

# 导入工具
from ml.utils.data_utils import read_data, generate_time_lags, time_to_feature, handle_nans, to_Xy, \
    to_torch_dataset, to_timeseries_rep, assign_statistics, \
    to_train_val, scale_features, get_data_by_area, remove_identifiers, get_exogenous_data_by_area, handle_outliers
from ml.utils.train_utils import train, test

# read_data 读取原始数据
# generate_time_lags 生成时间滞后
# time_to_feature 将时间转换为特征
# handle_nans 处理缺失值
# to_Xy 将数据转换为X和y
# to_torch_dataset 将数据转换为torch.Dataset
# to_timeseries_rep 将数据转换为时间序列表示
# assign_statistics 分配统计量
# train 训练模型
# test 测试模型

from ml.models.mlp import MLP
from ml.models.rnn import RNN
from ml.models.lstm import LSTM
from ml.models.gru import GRU
from ml.models.cnn import CNN
from ml.models.rnn_autoencoder import DualAttentionAutoEncoder



# 设置超参数

In [12]:
args = Namespace(
    data_path='../CBL-dataset/LCL-June2015v2_0.csv', # 电力数据集路径
    data_path_test=None, # 测试数据集（可选）
    test_size=0.2, # 验证集比例
    targets=['kwh'], # 目标列：用电量
    num_lags=10, # 用于输入的过去观测值数量

    
    filter_bs=None, # 是否使用单个客户进行训练，将在后续动态更改
    identifier='customer_id', # 标识客户的列名

    nan_constant=0,    # 用于转换NaN值的常数
    x_scaler='minmax', # X特征缩放器
    y_scaler='minmax', # y目标缩放器

    # ========= 可能会调整的 =========
    outlier_detection=None,  # 是否执行异常值处理（flooring and capping）

    
    criterion='mse', # 优化准则，mse或l1

    # ========= 可能会调整的 =========
    epochs=10,       # 最大训练轮数

    lr=0.001,        # 学习率
    optimizer='adam', # 优化器，可以是sgd或adam
    batch_size=128, # 批次大小

    # ========= 可能会调整的 =========
    early_stopping=True, # 是否使用早停
    patience=5, # 早停的耐心值（如果指定）

    max_grad_norm=0.0, # 是否裁剪梯度范数
    reg1=0.0, # l1正则化
    reg2=0.0, # l2正则化
    
    plot_history=True, # 绘制损失历史

    cuda=True, # 是否使用GPU
    seed=0, # 随机种子（可重现性）

    assign_stats=None, # 是否使用统计量作为外生数据, ["mean", "median", "std", "variance", "kurtosis", "skew"]
    use_time_features=False # 是否使用日期时间特征
)

print(f"Script arguments: {args}\n")

device = "cuda" if args.cuda and torch.cuda.is_available() else "cpu"
print(f"使用 {device} 训练")

Script arguments: Namespace(data_path='../CBL-dataset/LCL-June2015v2_0.csv', data_path_test=None, test_size=0.2, targets=['kwh'], num_lags=10, filter_bs=None, identifier='customer_id', nan_constant=0, x_scaler='minmax', y_scaler='minmax', outlier_detection=None, criterion='mse', epochs=10, lr=0.001, optimizer='adam', batch_size=128, early_stopping=True, patience=5, max_grad_norm=0.0, reg1=0.0, reg2=0.0, plot_history=True, cuda=True, seed=0, assign_stats=None, use_time_features=False)

使用 cuda 训练


# 设定随机种子

In [13]:
def seed_all():
    # ensure reproducibility
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all()

## 指定对某个特定客户进行异常值处理（参数控制是否启用）

In [14]:
# 默认不进行
if args.outlier_detection is not None:

    print("进行异常值处理。")

    # 配置需要处理的列。电力数据只有kwh列
    outlier_columns = ['kwh']  

    # 为不同客户设置不同的百分位数阈值，比如可以添加 {"MAC000002": (10, 90)}
    outlier_kwargs = {}  

    # 把上面的参数存入args
    args.outlier_columns = outlier_columns  # ['kwh']  
    args.outlier_kwargs = outlier_kwargs    # {}

else:
    print("没有进行异常值处理。")

没有进行异常值处理。


## 预处理（重要！！！！）

In [15]:
# # 详细说明版本（仅为分析目的，真正用的时候用下面定义的def）
# filter_bs = 'MAC000007'

# # 加载电力数据集
# df = pd.read_csv(args.data_path)

# # 清理列名首尾空格
# df.columns = df.columns.str.strip()

# # 重命名列
# df = df.rename(columns={
#     'LCLid': 'customer_id',
#     'stdorToU': 'tariff_type',
#     'DateTime': 'datetime',
#     'KWH/hh (per half hour)': 'kwh'
# })

# print("重命名列")
# display(df.head()) 

# # 转换数据类型，转换失败时用NaN
# df['datetime'] = pd.to_datetime(df['datetime'], format="%Y-%m-%d %H:%M:%S.%f", errors='coerce')
# df['kwh'] = pd.to_numeric(df['kwh'], errors='coerce')  

# print("转换数据类型")
# display(df.head())  

# # 删除tariff_type列（字符串类型，不适合建模）
# df = df.drop(['tariff_type'], axis=1)  # axis=1 表示按列删除
# print("删除tariff_type列")
# display(df.head())  

# # 设置时间索引
# # 将数据框（DataFrame）从一个“带有时间列的普通表”转换成了一个“以时间为坐标轴的时间序列（Time Series）对
# df.set_index(pd.DatetimeIndex(df["datetime"]), inplace=True)
# df.drop(["datetime"], axis=1, inplace=True)
# print("设置时间索引")
# display(df.head())  


# # 只保留customer_id和kwh列
# df = df[['customer_id', 'kwh']].copy()  # 使用copy的目的是在内存中专门开辟一块新空间存数据
# df['kwh'] = df['kwh'].astype("float32")
# print("只保留customer_id和kwh列")
# display(df.head())

# # 如果指定了filter_bs（客户ID），则过滤数据
# if filter_bs is not None:
#     # 使用loc，只把为True的结果拿出来
#     df = df.loc[df[args.identifier] == filter_bs]  # identifier指的是标识客户的列名
# print("只保留用户MAC000007")
# display(df.head())

# # 处理缺失值
# df = handle_nans(train_data=df,   # 训练数据
#                  constant=args.nan_constant,  # 缺失值填充常数，前面的超参数中定义为0
#                  identifier=args.identifier)  # 标识客户的列名（该列不处理缺失）
# print("处理缺失值")
# display(df.head())

# # 分割训练/验证集（传递正确的identifier参数）
# train_data, val_data = to_train_val(df, 
#                                     train_size=1-args.test_size,  # 训练集占80%
#                                     identifier=args.identifier) 
# print("分割【训练集】")
# display(train_data.head())
# print("训练集的行数：", len(train_data))
# print("================================================")
# print("分割【验证集】")
# display(val_data.head())
# print("验证集的行数：", len(val_data))
# print("================================================")

# # 处理异常值（如果指定）
# # 默认为None
# if args.outlier_detection is not None:
#     train_data = handle_outliers(df=train_data, 
#                                  columns=args.outlier_columns,  # ['kwh']  
#                                  identifier=args.identifier,    # 标识客户的列名
#                                  kwargs=args.outlier_kwargs)    # 异常值处理参数

# # 获取X和y（传递正确的identifier参数）
# # 这里的targets当做y，其他列作为X
# X_train, X_val, y_train, y_val = to_Xy(train_data=train_data,   # 刚刚分割的训练集
#                                        val_data=val_data,       # 刚刚分割的验证集
#                                        targets=args.targets,    # ['kwh']
#                                        identifier=args.identifier # 标识客户的列名
#                                 )
# print("这里是X")
# display(X_train.head())
# print("================================================")
# print("这里是y")
# display(y_train.head())

# # 缩放
# # 为什么需要缩放？
# # 在机器学习中，缩放（标准化/归一化）的主要目的：
# # 加速训练：神经网络和梯度下降算法在数据尺度相近时收敛更快
# # 避免数值问题：防止大数值导致梯度爆炸或消失
# # 公平对待特征：不同尺度的特征在模型中有相同的重要性
# # 
# # 下面缩放X
# X_train, X_val, x_scaler = scale_features(train_data=X_train, 
#                                           val_data=X_val,
#                                           scaler=args.x_scaler,    # x_scaler='minmax'
#                                           identifier=args.identifier
#                                           )
# # 缩放y
# y_train, y_val, y_scaler = scale_features(train_data=y_train, 
#                                           val_data=y_val,
#                                           scaler=args.y_scaler,  # y_scaler='minmax'
#                                           identifier=args.identifier
#                                           )
# print("================================================")
# print("缩放后的X")
# display(X_train.head())
# print("================================================")
# print("缩放后的y")
# display(y_train.head())
# print("================================================")


# # 生成时间滞后特征
# # 这里开始，X变为10维。Y变为每个窗口后面的下一个值
# # 例如：时序数据 1 2 3 4 5 6 7 8 9 10
# # X为：1 2 3 4 5 6 7 8 9
# # Y为：1 2 3 4 5 6 7 8 9
# # 处理后，变为（假设lags = 3，X变为3维）：
# # X为：[1 2 3], [2 3 4], [3 4 5], [4 5 6], [5 6 7], [6 7 8], [7 8 9]
# # Y为：  [4]  ,   [5]  ,   [6]  ,   [7]  ,   [8]  ,   [9]  ,   [10]
# X_train = generate_time_lags(X_train, args.num_lags, identifier=args.identifier)
# X_val   = generate_time_lags(X_val,   args.num_lags, identifier=args.identifier)
# y_train = generate_time_lags(y_train, args.num_lags, identifier=args.identifier, is_y=True)
# y_val   = generate_time_lags(y_val,   args.num_lags, identifier=args.identifier, is_y=True)

# print("生成时间滞后特征")
# display(X_train.head())
# print("================================================")
# display(y_train.head())
# print("================================================")


# # 获取日期时间特征作为外生数据，这里默认False，未启用
# # 流程：
# # 1. 从 DataFrame 的索引（时间索引）中提取 hour 和 minute
# # 2. 调用 to_cyclical 将时间组件转换为循环特征（让23:00 和 00:00 在循环空间中很接近）
# # 3. 返回外生特征
# date_time_df_train = time_to_feature(
#     X_train, args.use_time_features, identifier=args.identifier
# )
# date_time_df_val = time_to_feature(
#     X_val, args.use_time_features, identifier=args.identifier
# )

# # 获取统计量作为外生数据
# # 从时间滞后特征中计算统计量（如均值、标准差等），作为外生特征，用于描述历史时间窗口的整体特征。
# stats_df_train = assign_statistics(X_train, 
#                                    args.assign_stats,     # 之前设置为None，可以改为["mean", "median", "std", ...]
#                                    args.num_lags,         # 之前设置为10
#                                    targets=args.targets,  # 设置为["kwh"]
#                                    identifier=args.identifier  # 之前设置为"customer_id"
#                                    )

# stats_df_val = assign_statistics(X_val, args.assign_stats, args.num_lags, 
#                                     targets=args.targets, identifier=args.identifier)

# # 合并外生特征（如果有）
# # 按列水平合并，移除重复列
# if date_time_df_train is not None or stats_df_train is not None:
#     exogenous_data_train = pd.concat([date_time_df_train, stats_df_train], axis=1)
#     # 移除重复列（如果有）
#     exogenous_data_train = exogenous_data_train.loc[:, ~exogenous_data_train.columns.duplicated()].copy()
#     assert len(exogenous_data_train) == len(X_train) == len(y_train)
# else:
#     exogenous_data_train = None
# if date_time_df_val is not None or stats_df_val is not None:
#     exogenous_data_val = pd.concat([date_time_df_val, stats_df_val], axis=1)
#     exogenous_data_val = exogenous_data_val.loc[:, ~exogenous_data_val.columns.duplicated()].copy()
#     assert len(exogenous_data_val) == len(X_val) == len(y_val)
# else:
#     exogenous_data_val = None

# # 最终得到：    
# X_train, X_val, 
# y_train, y_val, 
# exogenous_data_train, 
# exogenous_data_val, 
# x_scaler, y_scaler

In [16]:
# filter_bs 制定需要预处理的客户
def make_preprocessing(filter_bs=None):
    """预处理电力数据集"""
    
    # 加载电力数据集
    df = pd.read_csv(args.data_path)
    
    # 清理列名首尾空格
    df.columns = df.columns.str.strip()
    
    # 重命名列
    df = df.rename(columns={
        'LCLid': 'customer_id',
        'stdorToU': 'tariff_type',
        'DateTime': 'datetime',
        'KWH/hh (per half hour)': 'kwh'
    })
    
    # 转换数据类型，转换失败时用NaN
    df['datetime'] = pd.to_datetime(df['datetime'], format="%Y-%m-%d %H:%M:%S.%f", errors='coerce')
    df['kwh'] = pd.to_numeric(df['kwh'], errors='coerce')  
    
    # 删除tariff_type列（字符串类型，不适合建模）
    df = df.drop(['tariff_type'], axis=1)  # axis=1 表示按列删除
    
    # 设置时间索引
    df.set_index(pd.DatetimeIndex(df["datetime"]), inplace=True)
    df.drop(["datetime"], axis=1, inplace=True)
    
    # 只保留customer_id和kwh列
    df = df[['customer_id', 'kwh']].copy()
    df['kwh'] = df['kwh'].astype("float32")
    
    # 如果指定了filter_bs（客户ID），则过滤数据
    if filter_bs is not None:
        df = df.loc[df[args.identifier] == filter_bs]
    
    # 处理缺失值
    df = handle_nans(train_data=df, constant=args.nan_constant,
                     identifier=args.identifier)
    
    # 分割训练/验证集（传递正确的identifier参数）
    train_data, val_data = to_train_val(df, train_size=1-args.test_size, identifier=args.identifier)
    
    # 处理异常值（如果指定）
    if args.outlier_detection is not None:
        train_data = handle_outliers(df=train_data, columns=args.outlier_columns,
                                     identifier=args.identifier, kwargs=args.outlier_kwargs)
    
    # 获取X和y（传递正确的identifier参数）
    X_train, X_val, y_train, y_val = to_Xy(train_data=train_data, val_data=val_data,
                                          targets=args.targets, identifier=args.identifier)
    
    # 缩放X
    X_train, X_val, x_scaler = scale_features(train_data=X_train, val_data=X_val,
                                             scaler=args.x_scaler, identifier=args.identifier)
    # 缩放y
    y_train, y_val, y_scaler = scale_features(train_data=y_train, val_data=y_val,
                                             scaler=args.y_scaler, identifier=args.identifier)
    
    # 生成时间滞后特征
    X_train = generate_time_lags(X_train, args.num_lags, identifier=args.identifier)
    X_val = generate_time_lags(X_val, args.num_lags, identifier=args.identifier)
    y_train = generate_time_lags(y_train, args.num_lags, identifier=args.identifier, is_y=True)
    y_val = generate_time_lags(y_val, args.num_lags, identifier=args.identifier, is_y=True)
    
    
    date_time_df_train = time_to_feature(
        X_train, args.use_time_features, identifier=args.identifier
    )
    date_time_df_val = time_to_feature(
        X_val, args.use_time_features, identifier=args.identifier
    )
    

    stats_df_train = assign_statistics(X_train, args.assign_stats, args.num_lags,
                                       targets=args.targets, identifier=args.identifier)
    stats_df_val = assign_statistics(X_val, args.assign_stats, args.num_lags, 
                                       targets=args.targets, identifier=args.identifier)
    
    # 合并外生特征（如果有）
    if date_time_df_train is not None or stats_df_train is not None:
        exogenous_data_train = pd.concat([date_time_df_train, stats_df_train], axis=1)
        # 移除重复列（如果有）
        exogenous_data_train = exogenous_data_train.loc[:, ~exogenous_data_train.columns.duplicated()].copy()
        assert len(exogenous_data_train) == len(X_train) == len(y_train)
    else:
        exogenous_data_train = None
    if date_time_df_val is not None or stats_df_val is not None:
        exogenous_data_val = pd.concat([date_time_df_val, stats_df_val], axis=1)
        exogenous_data_val = exogenous_data_val.loc[:, ~exogenous_data_val.columns.duplicated()].copy()
        assert len(exogenous_data_val) == len(X_val) == len(y_val)
    else:
        exogenous_data_val = None
        
    return X_train, X_val, y_train, y_val, exogenous_data_train, exogenous_data_val, x_scaler, y_scaler

## 正式进行预处理（需要耗费一定时间）

In [17]:
# 这里 exogenous_data_train 和 val 为 None（除非启用时间特征或统计量）
# 选择一个客户进行训练（例如数据量最多的客户之一）
X_train, X_val, y_train, y_val, exogenous_data_train, exogenous_data_val, x_scalers, y_scalers = make_preprocessing(
    # filter_bs="MAC000018"  # 选择一个客户ID，例如数据量最多的客户
)

display(X_train.head())
display(y_train.head())

# exogenous_data_train, exogenous_data_val 为None
# x_scaler 和 y_scaler 是缩放器（Scaler）对象，用于：
# 训练时：将数据从原始尺度缩放到模型需要的尺度（如 [0, 1]）
# 预测时：将模型输出从缩放尺度还原回原始尺度

INFO logger 2026-01-07 21:05:27,643 | data_utils.py:383 | Observations info in MAC000002
INFO logger 2026-01-07 21:05:27,644 | data_utils.py:384 | 	Total number of samples:  24158
INFO logger 2026-01-07 21:05:27,646 | data_utils.py:385 | 	Number of samples for training: 19327
INFO logger 2026-01-07 21:05:27,647 | data_utils.py:386 | 	Number of samples for validation:  4831
INFO logger 2026-01-07 21:05:27,716 | data_utils.py:383 | Observations info in MAC000003
INFO logger 2026-01-07 21:05:27,717 | data_utils.py:384 | 	Total number of samples:  35469
INFO logger 2026-01-07 21:05:27,718 | data_utils.py:385 | 	Number of samples for training: 28376
INFO logger 2026-01-07 21:05:27,720 | data_utils.py:386 | 	Number of samples for validation:  7093
INFO logger 2026-01-07 21:05:27,789 | data_utils.py:383 | Observations info in MAC000004
INFO logger 2026-01-07 21:05:27,790 | data_utils.py:384 | 	Total number of samples:  31677
INFO logger 2026-01-07 21:05:27,792 | data_utils.py:385 | 	Number of

,kwh_lag-10,kwh_lag-9,kwh_lag-8,kwh_lag-7,kwh_lag-6,kwh_lag-5,kwh_lag-4,kwh_lag-3,kwh_lag-2,kwh_lag-1,customer_id
datetime,,,,,,,,,,,
2012-10-12 05:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002
2012-10-12 06:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002
2012-10-12 06:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002
2012-10-12 07:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002
2012-10-12 07:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAC000002


,kwh,customer_id
datetime,,
2012-10-12 05:30:00,0.0,MAC000002
2012-10-12 06:00:00,0.0,MAC000002
2012-10-12 06:30:00,0.0,MAC000002
2012-10-12 07:00:00,0.0,MAC000002
2012-10-12 07:30:00,0.0,MAC000002


## 后处理阶段

为什么需要两个阶段？
- 职责分离：
    - 预处理：数据清洗和特征工程
    - 后处理：格式转换和形状重塑
- 灵活性：
    - 预处理可以多次调整特征
    - 后处理只需在训练前执行一次

In [18]:
def make_postprocessing(X_train, X_val, y_train, y_val, exogenous_data_train, exogenous_data_val, x_scaler, y_scaler):
    """Make data ready to be fed into ml algorithms"""
    # 按区域切分数据
    # 检查数据中是否包含多个区域（多个用户），不等于1，说明是混合数据。
    # get_data_by_area 将数据拆分成一个个字典dict，Key 是区域名，Value 是该区域对应的数据。
    # 这为联邦学习中“每个客户端拥有独立数据”做准备。
    if X_train[args.identifier].nunique() != 1:
        client_X_train, client_X_val, client_y_train, client_y_val = get_data_by_area(X_train, X_val,
                                                                              y_train, y_val, 
                                                                              identifier=args.identifier)
    else:
        client_X_train, client_X_val, client_y_train, client_y_val = None, None, None, None

    print("数据中包含的用户列表")
    print(client_X_train.keys())

    # 按客户分组【外生数据】
    if exogenous_data_train is not None:
        exogenous_data_train, exogenous_data_val = get_exogenous_data_by_area(exogenous_data_train,
                                                                              exogenous_data_val)
    # 转换为 NumPy 格式
    if client_X_train is not None:
        # area即用户（key value），这里对每个用户进行处理。
        for area in client_X_train:
            # 移除 identifier 列，client_X值从11维变为10维
            tmp_X_train, tmp_y_train, tmp_X_val, tmp_y_val = remove_identifiers(client_X_train[area], 
                                                                                client_y_train[area], 
                                                                                client_X_val[area], 
                                                                                client_y_val[area],
                                                                                identifier=args.identifier)
            
            # 将 Pandas 表格彻底转为纯数字矩阵（NumPy ndarray），深度学习框架（如 PyTorch）的输入必须是张量，而张量最方便的中间转换格式就是 NumPy。
            tmp_X_train, tmp_y_train = tmp_X_train.to_numpy(), tmp_y_train.to_numpy()
            tmp_X_val, tmp_y_val     = tmp_X_val.to_numpy(),   tmp_y_val.to_numpy()

            # 把处理好的结果存回字典
            client_X_train[area] = tmp_X_train
            client_X_val[area]   = tmp_X_val
            client_y_train[area] = tmp_y_train
            client_y_val[area]   = tmp_y_val
    
    # 将外生数据转为NumPy格式
    if exogenous_data_train is not None:
        for area in exogenous_data_train:
            exogenous_data_train[area] = exogenous_data_train[area].to_numpy()
            exogenous_data_val[area]   = exogenous_data_val[area].to_numpy()
    
    # 移除 identifier 列，X值从11维变为10维。此时X_train仍为pandas，不是numpy
    X_train, y_train, X_val, y_val = remove_identifiers(X_train, y_train, X_val, y_val,
                                                         identifier=args.identifier)
    assert len(X_train.columns) == len(X_val.columns)
    
    # 计算每个时间步的特征数（即在做时间滞后处理之前，你到底有多少个基础变量）
    num_features = len(X_train.columns) // args.num_lags  # 结果是1
    
    
    # 将数据重塑为时间序列形状 X_train.shape = (799714, 10, 1, 1) 样本数 时间步数 每个时间步的特征数 额外维度（在大多数情况下，这个维度是冗余的，但有些模型是4D的）
    X_train = to_timeseries_rep(X_train.to_numpy(), num_lags=args.num_lags,
                                            num_features=num_features)
    X_val = to_timeseries_rep(X_val.to_numpy(), num_lags=args.num_lags,
                                          num_features=num_features)

    if client_X_train is not None:
        client_X_train = to_timeseries_rep(client_X_train, num_lags=args.num_lags,
                                                     num_features=num_features)
        client_X_val = to_timeseries_rep(client_X_val, num_lags=args.num_lags,
                                                   num_features=num_features)
    
    # transform targets to numpy
    y_train, y_val = y_train.to_numpy(), y_val.to_numpy()
    
    # 集中式学习场景，将所有客户的外生数据合并为一个整体。默认这里不会进入。
    if not args.filter_bs and exogenous_data_train is not None:
        exogenous_data_train_combined, exogenous_data_val_combined = [], []

        for area in exogenous_data_train:
            exogenous_data_train_combined.extend(exogenous_data_train[area])
            exogenous_data_val_combined.extend(exogenous_data_val[area])

        exogenous_data_train_combined = np.stack(exogenous_data_train_combined)
        exogenous_data_val_combined = np.stack(exogenous_data_val_combined)

        # 定义一个新的 all 键，存储完整数据
        exogenous_data_train["all"] = exogenous_data_train_combined
        exogenous_data_val["all"] = exogenous_data_val_combined


    return X_train, X_val, y_train, y_val, client_X_train, client_X_val, client_y_train, client_y_val, exogenous_data_train, exogenous_data_val

In [19]:
X_train, X_val, y_train, y_val, client_X_train, client_X_val, client_y_train, client_y_val, exogenous_data_train, exogenous_data_val = make_postprocessing(X_train, X_val, y_train, y_val, exogenous_data_train, exogenous_data_val, x_scalers, y_scalers)

数据中包含的用户列表
dict_keys(['MAC000002', 'MAC000003', 'MAC000004', 'MAC000006', 'MAC000007', 'MAC000008', 'MAC000009', 'MAC000010', 'MAC000011', 'MAC000012', 'MAC000013', 'MAC000016', 'MAC000018', 'MAC000019', 'MAC000020', 'MAC000021', 'MAC000022', 'MAC000023', 'MAC000024', 'MAC000025', 'MAC000026', 'MAC000027', 'MAC000028', 'MAC000029', 'MAC000030', 'MAC000032', 'MAC000033', 'MAC000034', 'MAC000035', 'MAC000036'])


In [ ]:
print("X_train的shape", X_train.shape)
print("X_val的shap   ", X_val.shape)
print("y_train的shape", y_train.shape)
print("y_val的shape  ", y_val.shape)

print("client_X_train的keys", client_X_train.keys())
print("client_X_val的keys  ", client_X_val.keys())
print("client_y_train的keys", client_y_train.keys())
print("client_y_val的keys  ", client_y_val.keys())

print("用户MAC000007的shape", client_X_train["MAC000007"].shape)
print("用户MAC000007的shape", client_X_val["MAC000007"].shape)
print("用户MAC000007的shape", client_y_train["MAC000007"].shape)
print("用户MAC000007的shape", client_y_val["MAC000007"].shape)


X_train的shape (799714, 10, 1, 1)
X_val的shap    (199686, 10, 1, 1)
y_train的shape (799714, 1)
y_val的shape   (199686, 1)
client_X_train的keys dict_keys(['MAC000002', 'MAC000003', 'MAC000004', 'MAC000006', 'MAC000007', 'MAC000008', 'MAC000009', 'MAC000010', 'MAC000011', 'MAC000012', 'MAC000013', 'MAC000016', 'MAC000018', 'MAC000019', 'MAC000020', 'MAC000021', 'MAC000022', 'MAC000023', 'MAC000024', 'MAC000025', 'MAC000026', 'MAC000027', 'MAC000028', 'MAC000029', 'MAC000030', 'MAC000032', 'MAC000033', 'MAC000034', 'MAC000035', 'MAC000036'])
client_X_val的keys   dict_keys(['MAC000002', 'MAC000003', 'MAC000004', 'MAC000006', 'MAC000007', 'MAC000008', 'MAC000009', 'MAC000010', 'MAC000011', 'MAC000012', 'MAC000013', 'MAC000016', 'MAC000018', 'MAC000019', 'MAC000020', 'MAC000021', 'MAC000022', 'MAC000023', 'MAC000024', 'MAC000025', 'MAC000026', 'MAC000027', 'MAC000028', 'MAC000029', 'MAC000030', 'MAC000032', 'MAC000033', 'MAC000034', 'MAC000035', 'MAC000036'])
client_y_train的keys dict_keys(['MAC000

array([[[0.02987132]],

       [[0.02359069]],

       [[0.01807598]],

       [[0.0189951 ]],

       [[0.01317402]],

       [[0.01256127]],

       [[0.01746324]],

       [[0.01332721]],

       [[0.01041667]],

       [[0.01164216]]])

## 分析数据

In [21]:
for client in client_X_train:
    print(f"\nClient: {client}")
    print(f"X_train shape: {client_X_train[client].shape}, y_train shape: {client_y_train[client].shape}")
    print(f"X_val shape: {client_X_val[client].shape}, y_val shape: {client_y_val[client].shape}")


Client: MAC000002
X_train shape: (19317, 10, 1, 1), y_train shape: (19317, 1)
X_val shape: (4821, 10, 1, 1), y_val shape: (4821, 1)

Client: MAC000003
X_train shape: (28366, 10, 1, 1), y_train shape: (28366, 1)
X_val shape: (7083, 10, 1, 1), y_val shape: (7083, 1)

Client: MAC000004
X_train shape: (25332, 10, 1, 1), y_train shape: (25332, 1)
X_val shape: (6325, 10, 1, 1), y_val shape: (6325, 1)

Client: MAC000006
X_train shape: (29159, 10, 1, 1), y_train shape: (29159, 1)
X_val shape: (7282, 10, 1, 1), y_val shape: (7282, 1)

Client: MAC000007
X_train shape: (20027, 10, 1, 1), y_train shape: (20027, 1)
X_val shape: (4999, 10, 1, 1), y_val shape: (4999, 1)

Client: MAC000008
X_train shape: (20801, 10, 1, 1), y_train shape: (20801, 1)
X_val shape: (5192, 10, 1, 1), y_val shape: (5192, 1)

Client: MAC000009
X_train shape: (20181, 10, 1, 1), y_train shape: (20181, 1)
X_val shape: (5037, 10, 1, 1), y_val shape: (5037, 1)

Client: MAC000010
X_train shape: (20030, 10, 1, 1), y_train shape: (